# 52 — Render the Dataset Specialization A-Box

The layer decision D4 deferred. Decision D25 fixes what it is: a mechanistic
rendering of the published Dataset Specializations, nothing authored on top.

Output: one file per domain, `dss/cosmos_sdtm_v1.{DOMAIN}.instances.ttl`,
32 files and no combined one (decision D32), each canonicalized per decision D9.

Rendered directly with `rdflib` rather than through `linkml-convert` (decision
D8), as `50_render_bc.ipynb` does for the concept layer.

**What is not rendered:** nothing at specialization grain — all 1,475
specializations have an IRI under decision D3 and every one renders. The
counterpart of the six concepts D2 omits is five *references* that resolve to no
node: four specializations pointing at `NEW_` concepts and one variable pointing
at `NEW_DEC1`. Those carry the published string as a literal (decision D31) and
are reported below.

Four authored shapes inside the core rendering, each named in
`docs/decisions.md` rather than left implicit: the variable IRI (D26), the
variable order carried as `rdf:_n` membership (D27), the assigned term as a node
per use (D28) and the codelist as a shared node (D29). All mint IRIs under this
repo's namespace on the pattern `50_` set with D18 and D21.

## Configuration

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
REPORTS   = "../reports"
OUT_DIR   = "../dss"

DSS_EXPORT = f"{DOWNLOADS}/cdisc_sdtm_dataset_specializations_latest.csv"
BC_EXPORT  = f"{DOWNLOADS}/cdisc_biomedical_concepts_latest.csv"
SDTM_TBOX  = f"{ROOT}/cosmos_sdtm_v1.ttl"

SDTM_NS = "https://www.cdisc.org/cosmos/sdtm_v1.0/"
OBO     = "http://purl.obolibrary.org/obo/NCIT_"
EVS     = "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"

VERSION = "0.4.0"
ONTOLOGY_IRI = "https://w3id.org/cdisc/cosmos/sdtm/"

# D3: a specialization is domain-scoped under this repo's namespace. D26: its
# variables sit under it. D28 and D30: a variable's assigned term and
# relationship sit under the variable. All minted; none names anything CDISC
# named.
DSS_NS = "https://w3id.org/cdisc/cosmos/dss/"

## The specialization-level view

The flat export is at variable grain: 13,922 rows for 1,475 specializations,
with specialization-level values repeated on every row. A specialization
rebuilds by grouping on `(domain, vlm_group_id)`.

Unlike the concept export, no latest-`package_date` rule is needed: measured at
this pin, every group carries exactly one `package_date`. That is asserted
rather than assumed — if a group ever carries two, the rule for choosing is not
settled and this notebook stops.

The remaining preconditions are the ones decision D5 measured: every
specialization-level column is constant within its group, `(specialization,
variable)` is unique so each row is exactly one `SDTMVariable`, and the order
of `variables` is the row order — the one dependency on the export that is not
a value, stated in each file's provenance. The identifier patterns are checked
too, because the D3 and D26 IRIs compose them unescaped.

In [ ]:
from pathlib import Path

import pandas as pd

export = pd.read_csv(DSS_EXPORT, dtype=str, keep_default_na=False)

GROUP = ["domain", "vlm_group_id"]
GROUP_COLUMNS = ["package_date", "bc_id", "sdtmig_start_version", "sdtmig_end_version",
                 "vlm_source", "short_name"]
NAME_PATTERN = r"[A-Z][A-Z0-9_]*"

if (export.groupby(GROUP)[GROUP_COLUMNS].nunique() > 1).any().any():
    raise RuntimeError("D5: a specialization-level column varies within its group")
if export.duplicated(GROUP + ["sdtm_variable"]).any():
    raise RuntimeError("D5: (specialization, variable) is not unique")
if not export.vlm_group_id.str.fullmatch(NAME_PATTERN).all():
    raise RuntimeError("a specialization identifier does not match the model pattern")
if not export.sdtm_variable.str.fullmatch(NAME_PATTERN).all():
    raise RuntimeError("a variable name does not match the model pattern")

# D5: position within the group is the export row order, 1-based.
export["position"] = export.groupby(GROUP).cumcount() + 1

groups = export.drop_duplicates(GROUP).set_index(GROUP)
domains = sorted(export.domain.unique())

print(f"export rows              {len(export):>6,}")
print(f"specializations          {len(groups):>6,}")
print(f"domains                  {len(domains):>6,}")
print(f"max variables per group  {export.position.max():>6,}   (D27: the sh:ignoredProperties bound)")

## Identity, and the references that resolve to nothing

A specialization's IRI is `…/cosmos/dss/{DOMAIN}/{MNEMONIC}` (decision D3), a
variable's is that plus `/{VARIABLE}` (decision D26). The mnemonic is carried as
`dcterms:identifier`, as `50_` carries the bare C-code: the identifier is the
IRI, not a property repeated beside it.

`biomedicalConceptId` and `dataElementConceptId` are edges to the NCIt nodes
the concept layer declares (decision D31, on the D21 pattern), so the concept
export is read here to resolve them. Every `bc_id` and every `dec_id` in this
export exists there — asserted. Five references exist there without an NCIt
code, all `NEW_` placeholders: four specializations and one variable. Decision
D2 mints nothing for a placeholder, and dropping the reference would lose
published information, so the published string is carried as a literal on the
same property. The RDF term type is the whole marker — `FILTER(isLiteral(?o))`
finds every one — and nothing is minted to flag it. They are written to
`reports/dss_unresolved_references.csv` so a change at the next pin shows as a
diff.

In [ ]:
import csv

from rdflib import URIRef

bc_export = pd.read_csv(BC_EXPORT, dtype=str, keep_default_na=False)

latest = bc_export.package_date.groupby(bc_export.bc_id).transform("max")
bc_code = bc_export[bc_export.package_date == latest].drop_duplicates("bc_id").set_index("bc_id").ncit_code

dec_rows = bc_export[bc_export.dec_id != ""]
if (dec_rows.groupby("dec_id").ncit_dec_code.nunique() > 1).any():
    raise RuntimeError("a DEC identifier maps to more than one NCIt code")
dec_code = dec_rows.drop_duplicates("dec_id").set_index("dec_id").ncit_dec_code

missing = sorted(set(groups.bc_id) - set(bc_code.index))
if missing:
    raise RuntimeError(f"bc_id not in the concept export: {missing}")
missing = sorted(set(export.dec_id) - {""} - set(dec_code.index))
if missing:
    raise RuntimeError(f"dec_id not in the concept export: {missing}")


def dss_iri(domain, mnemonic):
    return URIRef(f"{DSS_NS}{domain}/{mnemonic}")


def var_iri(domain, mnemonic, name):
    return URIRef(f"{DSS_NS}{domain}/{mnemonic}/{name}")


def concept_iri(code):
    return URIRef(OBO + code)


unresolved = []
for (domain, mnemonic), row in groups[groups.bc_id.map(bc_code) == ""].iterrows():
    unresolved.append({"kind": "bc", "domain": domain, "specialization": mnemonic,
                       "variable": "", "reference": row.bc_id})
for _, row in export[(export.dec_id != "") & (export.dec_id.map(dec_code) == "")].iterrows():
    unresolved.append({"kind": "dec", "domain": row.domain, "specialization": row.vlm_group_id,
                       "variable": row.sdtm_variable, "reference": row.dec_id})

if not all(u["reference"].startswith("NEW_") for u in unresolved):
    raise RuntimeError("a reference lacks an NCIt code without being a NEW_ placeholder")

report = Path(REPORTS, "dss_unresolved_references.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["kind", "domain", "specialization", "variable", "reference"])
    writer.writeheader()
    writer.writerows(unresolved)

print(f"wrote {report}: {len(unresolved)} references carried as literals (D31)")
for u in unresolved:
    print(f"   {u['kind']:3}  {u['domain']}.{u['specialization']}{'.' + u['variable'] if u['variable'] else '':<32} {u['reference']}")

## Resolving enum values, booleans and integers

Property IRIs come from the T-Box, so the A-Box and the ontology use one
vocabulary. Enum-ranged values resolve to the permissible-value IRIs the T-Box
declares, but not by composing `{Enum}#{value}` as `50_` does — two enums here
break that. `LinkingPhraseEnum` members are percent-encoded in their IRIs
(`…#assesses%20seriousness%20of`), and `OriginTypeEnum` and `OriginSourceEnum`
members are NCIt classes (`NCIT:C170547` for `Assigned`) because the published
model gives each value a `meaning:`. What is uniform across all eight enums is
`rdfs:label`, so the resolver reads `linkml:permissible_values` off each enum
and keys on the label. A value the T-Box does not declare stops the notebook.

Booleans arrive as `Y`/`N` and are always filled, except `vlm_target`, which is
`Y` or empty — empty is omitted, not coerced to false, since the slot is optional.
`length` and `significant_digits` arrive as `200.0` where the model says integer;
they are cast, asserting integrality (decision D32, `known-gaps.md`).

In [ ]:
from rdflib import BNode, Graph, Literal, Namespace
from rdflib.namespace import DCTERMS, OWL, RDF, RDFS, SKOS, XSD

SDTM   = Namespace(SDTM_NS)
LINKML = Namespace("https://w3id.org/linkml/")

tbox = Graph().parse(SDTM_TBOX, format="turtle")

ENUMS = {}
for enum_name in ("PackageTypeEnum", "RoleEnum", "ComparatorEnum", "SDTMVariableDataTypeEnum",
                  "OriginTypeEnum", "OriginSourceEnum", "LinkingPhraseEnum", "PredicateTermEnum"):
    members = {str(tbox.value(pv, RDFS.label)): pv
               for pv in tbox.objects(SDTM[enum_name], LINKML.permissible_values)}
    if not members:
        raise RuntimeError(f"{enum_name} declares no permissible values in the T-Box")
    ENUMS[enum_name] = members


def enum_value(enum_name, value):
    """Permissible-value IRI as declared in the T-Box, by label; fail if it is not there."""
    if value not in ENUMS[enum_name]:
        raise RuntimeError(f"{value!r} is not a permissible value of {enum_name}")
    return ENUMS[enum_name][value]


def boolean(flag):
    if flag not in ("Y", "N"):
        raise RuntimeError(f"{flag!r} is not a Y/N flag")
    return Literal(flag == "Y")


def integer(value):
    number = float(value)
    if number != int(number):
        raise RuntimeError(f"{value!r} is not integral where the model says integer")
    return Literal(int(number))


def split(value):
    return [part.strip() for part in value.split(";") if part.strip()]


for enum_name, members in ENUMS.items():
    print(f"{enum_name:26} {len(members):>3} permissible values")

## Render the specializations

One `SDTMGroup` per group. `source` is the export's `vlm_source`;
`packageType` is not in the export and is set to the enum's single value,
`sdtm`, as `50_` sets `bc` (decision D32). `sdtmigEndVersion` is optional in the
model and emitted only where filled.

In [ ]:
g = Graph()
bc_literals = 0

for (domain, mnemonic), row in groups.iterrows():
    subject = dss_iri(domain, mnemonic)

    g.add((subject, RDF.type, SDTM.SDTMGroup))
    g.add((subject, DCTERMS.identifier, Literal(mnemonic)))

    g.add((subject, SDTM.domain, Literal(domain)))
    g.add((subject, SDTM.shortName, Literal(row.short_name)))
    g.add((subject, SDTM.source, Literal(row.vlm_source)))
    g.add((subject, SDTM.sdtmigStartVersion, Literal(row.sdtmig_start_version)))
    if row.sdtmig_end_version:
        g.add((subject, SDTM.sdtmigEndVersion, Literal(row.sdtmig_end_version)))
    g.add((subject, SDTM.packageDate, Literal(row.package_date, datatype=XSD.date)))
    g.add((subject, SDTM.packageType, enum_value("PackageTypeEnum", "sdtm")))

    # D31: an edge to the concept node, or the published string where there is none.
    code = bc_code[row.bc_id]
    if code:
        g.add((subject, SDTM.biomedicalConceptId, concept_iri(code)))
    else:
        g.add((subject, SDTM.biomedicalConceptId, Literal(row.bc_id)))
        bc_literals += 1

print(f"{len(g):,} triples after specialization headers; {bc_literals} biomedicalConceptId literals (D31)")

## Render the variables

One `SDTMVariable` per row, at its D26 IRI. The specialization reaches it twice:
by `variables`, which is exactly what the T-Box declares, and by `rdf:_n`, which
carries the D5 position in standard RDF vocabulary with nothing minted
(decision D27).

`dataElementConceptId` is an edge to the shared DEC node that
`cosmos_bc_v1.instances.ttl` declares — the same node the concept's use-node
reaches by `conceptId` under D21 — or the published string for the one `NEW_DEC1`
(D31). `valueList` is the only `;`-joined column that is split: one
`assigned_value` legitimately contains a `;` and is never touched.

Seven rows carry no `role`; the slot is optional and they render without one.
That is the export as published, not an anomaly — they are listed in the
confirm cell and nowhere else.

In [ ]:
RDF_MEMBER = str(RDF) + "_"

dec_literals = 0
roleless = []

for _, row in export.iterrows():
    subject = dss_iri(row.domain, row.vlm_group_id)
    variable = var_iri(row.domain, row.vlm_group_id, row.sdtm_variable)

    g.add((subject, SDTM.variables, variable))
    g.add((subject, URIRef(f"{RDF_MEMBER}{row.position}"), variable))

    g.add((variable, RDF.type, SDTM.SDTMVariable))
    g.add((variable, SDTM.name, Literal(row.sdtm_variable)))

    if row.dec_id:
        code = dec_code[row.dec_id]
        if code:
            g.add((variable, SDTM.dataElementConceptId, concept_iri(code)))
        else:
            g.add((variable, SDTM.dataElementConceptId, Literal(row.dec_id)))
            dec_literals += 1

    g.add((variable, SDTM.isNonStandard, boolean(row.nsv_flag)))
    for value in split(row.value_list):
        g.add((variable, SDTM.valueList, Literal(value)))
    if row.role:
        g.add((variable, SDTM.role, enum_value("RoleEnum", row.role)))
    else:
        roleless.append(f"{row.domain}.{row.vlm_group_id}.{row.sdtm_variable}")
    if row.data_type:
        g.add((variable, SDTM.dataType, enum_value("SDTMVariableDataTypeEnum", row.data_type)))
    if row.length:
        g.add((variable, SDTM.length, integer(row.length)))
    if row.format:
        g.add((variable, SDTM["format"], Literal(row.format)))
    if row.significant_digits:
        g.add((variable, SDTM.significantDigits, integer(row.significant_digits)))
    g.add((variable, SDTM.mandatoryVariable, boolean(row.mandatory_variable)))
    g.add((variable, SDTM.mandatoryValue, boolean(row.mandatory_value)))
    if row.origin_type:
        g.add((variable, SDTM.originType, enum_value("OriginTypeEnum", row.origin_type)))
    if row.origin_source:
        g.add((variable, SDTM.originSource, enum_value("OriginSourceEnum", row.origin_source)))
    if row.comparator:
        g.add((variable, SDTM.comparator, enum_value("ComparatorEnum", row.comparator)))
    if row.vlm_target:
        g.add((variable, SDTM.vlmTarget, boolean(row.vlm_target)))
    if row.subset_codelist:
        g.add((variable, SDTM.subsetCodelist, Literal(row.subset_codelist)))

print(f"{len(g):,} triples after variables; {dec_literals} dataElementConceptId literal (D31); {len(roleless)} without role")

## Codelists

**A codelist is a shared node at its NCIt IRI** (decision D29). A CDISC
codelist is an NCIt subset, so the code identifies it the way it identifies a
concept under D2, with the same dual anchor. `submissionValue` sits on the
shared node — asserted constant per code, since a value that varied by use
would belong to the pair instead, as D21 and D28 found for other slots. `href`
is not in the export and is not emitted. `subsetCodelist` stays a literal on
the variable: the model types the slot `string` and the export carries the
short name only — measured, it never occurs without a `codelist`.

In [ ]:
bound = export[export.codelist != ""]

if not bound.codelist.str.fullmatch(r"C[0-9]+").all():
    raise RuntimeError("a codelist reference is not a C-code")
if (bound.groupby("codelist").codelist_submission_value.nunique() > 1).any():
    raise RuntimeError("D29: a codelist carries more than one submissionValue across its uses")
if ((export.subset_codelist != "") & (export.codelist == "")).any():
    raise RuntimeError("a subsetCodelist occurs without a codelist")

for _, row in bound.iterrows():
    variable = var_iri(row.domain, row.vlm_group_id, row.sdtm_variable)
    g.add((variable, SDTM.codelist, concept_iri(row.codelist)))

for _, row in bound.drop_duplicates("codelist").iterrows():
    node = concept_iri(row.codelist)
    g.add((node, RDF.type, SDTM.CodeList))
    g.add((node, SKOS.exactMatch, URIRef(EVS + row.codelist)))
    g.add((node, DCTERMS.identifier, Literal(row.codelist)))
    g.add((node, SDTM.submissionValue, Literal(row.codelist_submission_value)))

codelists = bound.codelist.nunique()
print(f"codelist edges     {len(bound):>6,}")
print(f"codelist nodes     {codelists:>6,}   (D29)")

## Assigned terms

**An assigned term is a node per use, under the variable** (decision D28).
Measured at this pin, 951 of 1,307 distinct term codes carry more than one
`value` across the variables they are assigned to, so `value` is a property of
the (variable, term) pair — the D21 shape again. The node sits at
`{variable}/assignedTerm`; the slot is single-valued, so a fixed segment
identifies it, and it cannot be keyed on the code because 476 uses have a value
and no code, which the schema permits.

`conceptId` is an edge to the NCIt term. **Nothing is written onto the NCIt
node.** The schema declares no class for the term — unlike the DEC and the
codelist, there is nothing to type it as — so this is the line decision D22
drew for specimens: reach the shared IRI, assert nothing on it.

In [ ]:
terms = export[(export.assigned_term != "") | (export.assigned_value != "")]

if ((terms.assigned_term != "") & (terms.assigned_value == "")).any():
    raise RuntimeError("an assigned term has no value; the schema requires one")
if not terms.assigned_term[terms.assigned_term != ""].str.fullmatch(r"C[0-9]+").all():
    raise RuntimeError("an assigned term reference is not a C-code")

value_only = 0
for _, row in terms.iterrows():
    variable = var_iri(row.domain, row.vlm_group_id, row.sdtm_variable)
    node = URIRef(f"{variable}/assignedTerm")

    g.add((variable, SDTM.assignedTerm, node))
    g.add((node, RDF.type, SDTM.AssignedTerm))
    g.add((node, SDTM.value, Literal(row.assigned_value)))
    if row.assigned_term:
        g.add((node, SDTM.conceptId, concept_iri(row.assigned_term)))
    else:
        value_only += 1

print(f"assigned term nodes  {len(terms):>6,}   (D28)")
print(f"   with a concept    {len(terms) - value_only:>6,}")
print(f"   value only        {value_only:>6,}")

## Relationships

**Rendered as published** (decision D30): a node at `{variable}/relationship`
with the four fields, `linkingPhrase` and `predicateTerm` resolved to the
T-Box permissible values, `subject` and `object` as literals. The fields are
all filled or all empty — asserted — and `subject` equals the variable's own
name in every filled row, also asserted.

68 rows name an `object` that is not a variable of their own specialization.
The literal carries the name exactly as published, so nothing is lost; no edge
to a sibling node is derived, because the derivation would hide precisely the
rows a curator needs to see. They are written to
`reports/dss_unresolved_relationship_objects.csv` — curator input, diffable at
the next pin.

In [ ]:
fields = export[["subject", "linking_phrase", "predicate_term", "object"]]
filled = (fields != "").all(axis=1)
empty = (fields == "").all(axis=1)

if not (filled | empty).all():
    raise RuntimeError("D30: a relationship is partially filled")
if (export.subject[filled] != export.sdtm_variable[filled]).any():
    raise RuntimeError("D30: a relationship subject is not the variable's own name")

members = export.groupby(GROUP).sdtm_variable.apply(set)

unresolved_objects = []
for _, row in export[filled].iterrows():
    variable = var_iri(row.domain, row.vlm_group_id, row.sdtm_variable)
    node = URIRef(f"{variable}/relationship")

    g.add((variable, SDTM.relationship, node))
    g.add((node, RDF.type, SDTM.RelationShip))
    g.add((node, SDTM.subject, Literal(row.subject)))
    g.add((node, SDTM.linkingPhrase, enum_value("LinkingPhraseEnum", row.linking_phrase)))
    g.add((node, SDTM.predicateTerm, enum_value("PredicateTermEnum", row.predicate_term)))
    g.add((node, SDTM.object, Literal(row.object)))

    if row.object not in members[(row.domain, row.vlm_group_id)]:
        unresolved_objects.append({"domain": row.domain, "specialization": row.vlm_group_id,
                                   "variable": row.sdtm_variable, "linking_phrase": row.linking_phrase,
                                   "predicate_term": row.predicate_term, "object": row.object})

report = Path(REPORTS, "dss_unresolved_relationship_objects.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(unresolved_objects[0]))
    writer.writeheader()
    writer.writerows(unresolved_objects)

relationships = int(filled.sum())
print(f"relationship nodes   {relationships:>6,}   (D30)")
print(f"wrote {report}: {len(unresolved_objects)} objects not a variable of their own specialization, "
      f"{len({u['object'] for u in unresolved_objects})} distinct")

## Per-domain headers and canonical write

One file per domain (decision D32). Each domain's graph is the closure of
outgoing edges from that domain's specialization nodes, which also checks that
every triple rendered above hangs off a specialization — a triple reached from
no domain is an error. A codelist node used in several domains is repeated in
each file that uses it: same IRI, same triples, one node on load.

Each file carries its own header, `owl:imports`-ing the T-Box so a consumer
loading one gets the other (decisions D7, D9). The concept and DEC nodes that
`biomedicalConceptId` and `dataElementConceptId` reach are declared in the
concept instance graph and are **not** imported — dereferencing one variable IRI
must not pull the concept layer along. The header states that, the D5 row-order
dependency, and the two normalisations, as rules rather than counts: the counts
live in `reports/` and the confirm cell, where they are re-measured on every run.

In [ ]:
import json

from rdflib.compare import isomorphic, to_canonical_graph

meta = json.loads(Path(DOWNLOADS, ".fetch_meta_dss_export.json").read_text(encoding="utf-8"))
Path(OUT_DIR).mkdir(exist_ok=True)

HEADER_COMMENT = (
    "Dataset Specialization instances for domain {domain}, rendered from the pinned COSMoS export. "
    "Variable order is the export's row order, carried as rdf:_n membership beside the variables edges. "
    "length and significantDigits are cast from the export's decimal form to integer; "
    "packageType is not in the export and is set to the enum's single value. "
    "A biomedicalConceptId or dataElementConceptId that resolves to no node is carried as the published string. "
    "The concept nodes those edges reach are declared in the Biomedical Concept instance graph, which is not imported. "
    "Draft - not a normative CDISC artifact.")


def closure(graph, roots):
    """Every triple reachable by outgoing edges from the roots."""
    seen = set()
    frontier = list(roots)
    part = Graph()
    while frontier:
        node = frontier.pop()
        if node in seen:
            continue
        seen.add(node)
        for triple in graph.triples((node, None, None)):
            part.add(triple)
            if isinstance(triple[2], URIRef):
                frontier.append(triple[2])
    return part


covered = set()
written = {}

for domain in domains:
    part = closure(g, [dss_iri(domain, mnemonic) for mnemonic in groups.loc[domain].index])
    covered |= set(part)

    ontology = URIRef(f"{DSS_NS}{domain}")
    part.add((ontology, RDF.type, OWL.Ontology))
    part.add((ontology, OWL.imports, URIRef(ONTOLOGY_IRI)))
    part.add((ontology, RDFS.label, Literal(f"CDISC COSMoS Dataset Specializations, {domain} (instances)")))
    part.add((ontology, RDFS.comment, Literal(HEADER_COMMENT.format(domain=domain))))
    part.add((ontology, DCTERMS.source, URIRef(meta["raw_url"])))
    part.add((ontology, DCTERMS.identifier, Literal(meta["package_date"])))
    part.add((ontology, OWL.versionIRI, URIRef(f"{DSS_NS}{domain}/{VERSION}")))
    part.add((ontology, OWL.versionInfo, Literal(f"v{VERSION}")))

    canonical = to_canonical_graph(part)
    if not isomorphic(canonical, part) or len(canonical) != len(part):
        raise RuntimeError(f"{domain}: canonicalization changed the graph")

    out = Graph()
    for triple in canonical:
        out.add(triple)
    out.bind("cosmos_sdtm", SDTM_NS)
    out.bind("dcterms", DCTERMS)
    out.bind("skos", SKOS)
    out.bind("NCIT", OBO)
    out.bind("rdf", RDF)

    target = Path(OUT_DIR, f"cosmos_sdtm_v1.{domain}.instances.ttl")
    turtle = out.serialize(format="turtle")
    target.write_text(turtle, encoding="utf-8")
    written[domain] = len(out)
    print(f"{target.name:40} {len(out):>8,} triples  {len(turtle):>10,} chars")

if covered != set(g):
    raise RuntimeError(f"{len(set(g) - covered):,} triples reachable from no specialization")

print(f"\n{len(written)} files; {sum(written.values()):,} triples written for {len(g):,} rendered "
      f"(the difference is codelist nodes repeated across domains)")

## Confirm

Every file is re-read and its counts checked against the frame, so what is on
disk is what was rendered. The specialization-grain oddities are counted here
and nowhere else: they are the export as published and the schema permits them.

In [ ]:
reloaded = Graph()
for domain in domains:
    part = Graph().parse(Path(OUT_DIR, f"cosmos_sdtm_v1.{domain}.instances.ttl"), format="turtle")
    if len(part) != written[domain]:
        raise RuntimeError(f"{domain}: file does not reload to the triple count written")
    reloaded += part

specializations = set(reloaded.subjects(RDF.type, SDTM.SDTMGroup))
variables = set(reloaded.subjects(RDF.type, SDTM.SDTMVariable))
term_nodes = set(reloaded.subjects(RDF.type, SDTM.AssignedTerm))
relationship_nodes = set(reloaded.subjects(RDF.type, SDTM.RelationShip))
codelist_nodes = set(reloaded.subjects(RDF.type, SDTM.CodeList))
variable_edges = list(reloaded.subject_objects(SDTM.variables))
member_edges = [(s, p, o) for s, p, o in reloaded if str(p).startswith(RDF_MEMBER)]
bc_edges = list(reloaded.objects(None, SDTM.biomedicalConceptId))
dec_edges = list(reloaded.objects(None, SDTM.dataElementConceptId))

print(f"SDTMGroup nodes        {len(specializations):>7,}   (expected {len(groups):,})")
print(f"SDTMVariable nodes     {len(variables):>7,}   (expected {len(export):,})")
print(f"variables edges        {len(variable_edges):>7,}   rdf:_n edges {len(member_edges):,} (D27)")
print(f"AssignedTerm nodes     {len(term_nodes):>7,}   (D28; expected {len(terms):,})")
print(f"RelationShip nodes     {len(relationship_nodes):>7,}   (D30; expected {relationships:,})")
print(f"CodeList nodes         {len(codelist_nodes):>7,}   (D29; expected {codelists:,})")
print(f"blank nodes            {len({s for s in reloaded.subjects() if isinstance(s, BNode)}):>7,}")
print(f"ontology headers       {len(set(reloaded.subjects(RDF.type, OWL.Ontology))):>7,}")

if len(specializations) != len(groups):
    raise RuntimeError("specialization count does not match the frame")
if len(variables) != len(export):
    raise RuntimeError("variable count does not match the frame")
if len(member_edges) != len(variable_edges) or {o for _, _, o in member_edges} != {o for _, o in variable_edges}:
    raise RuntimeError("D27: rdf:_n membership does not mirror the variables edges")
if len(term_nodes) != len(terms) or len(relationship_nodes) != relationships:
    raise RuntimeError("child node counts do not match the frame")
if len(codelist_nodes) != codelists:
    raise RuntimeError("D29: codelist node count does not match the frame")
if any(isinstance(s, BNode) for s in reloaded.subjects()):
    raise RuntimeError("a blank node was written")
if any((c, SDTM.conceptId, None) in reloaded for c in reloaded.objects(None, SDTM.conceptId)):
    raise RuntimeError("D28: a triple was written onto an assigned term's NCIt node")

print()
print(f"published as-is, schema-permitted:  value-only assigned terms {value_only:,}; variables without role {len(roleless)}")
for name in roleless:
    print(f"   {name}")
print(f"references carried as literals (D31): biomedicalConceptId {sum(isinstance(o, Literal) for o in bc_edges)}, "
      f"dataElementConceptId {sum(isinstance(o, Literal) for o in dec_edges)}")
print(f"relationship objects not in own specialization (D30): {len(unresolved_objects)}")